In [1]:
import os 
import numpy as np
import pandas as pd
import math
import random
from dataclasses import dataclass
from typing import List, Dict, Any, Optional

import torch
from torch.utils.data import Dataset
from transformers import (
    T5Config,
    T5ForConditionalGeneration,
    PreTrainedTokenizerFast,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq,
)

# 1. Load data

In [2]:
PATH_DATA_FILE = os.path.join(os.getcwd(), 'dataset', 'processed', 'processed_data.csv')
PATH_TOKENIZER_FILE = os.path.join(os.getcwd(), 'python_tokenizer.json')

In [3]:
df = pd.read_csv(PATH_DATA_FILE)
list_python_function = df['method_code'].tolist()

list_python_function = list_python_function[:10_000]  # Limiting to first 1000 functions for testing

print(f"Total functions: {len(list_python_function)}")

Total functions: 10000


In [4]:
tokenizer = PreTrainedTokenizerFast(tokenizer_file=PATH_TOKENIZER_FILE)
print("Tokenizer loaded successfully.")

Tokenizer loaded successfully.


In [5]:
# T5 expects: pad/eos/unk + sentinel tokens <extra_id_0> ... <extra_id_99>
SPECIAL_SENTINELS = [f"<extra_id_{i}>" for i in range(100)]
added = False

specials = {
    "eos_token": "</s>",
    "unk_token": "<unk>",
    "pad_token": "<pad>",
}
for k, v in specials.items():
    if getattr(tokenizer, k, None) != v:
        setattr(tokenizer, k, v)
        added = True

# Add sentinel tokens to vocab if missing
missing_sentinels = [s for s in SPECIAL_SENTINELS if s not in tokenizer.get_vocab()]
if missing_sentinels:
    tokenizer.add_tokens(missing_sentinels, special_tokens=True)
    added = True
    
if tokenizer.pad_token is None:
    tokenizer.pad_token = "<pad>"

# 2. Config model

In [6]:
config = T5Config(
    vocab_size=len(tokenizer),  # critical: align with your tokenizer
    d_model=512,
    d_ff=2048,
    num_layers=8,
    num_heads=8,
    dropout_rate=0.1,
)
model = T5ForConditionalGeneration(config)
model.config.decoder_start_token_id = tokenizer.pad_token_id
print("Model initialized successfully.")

Model initialized successfully.


In [7]:
def _random_spans_noise_mask(length: int, noise_density=0.15, mean_span_length=3.0):
    """Return a boolean mask for which tokens are masked (True = masked)."""
    num_noise_tokens = max(1, min(length - 1, int(round(length * noise_density))))
    num_spans = max(1, int(round(num_noise_tokens / mean_span_length)))

    # Randomly partition the sequence into (noise and non-noise) spans
    # We build span lengths via a simple geometric-like process
    def _random_segmentation(num_items, num_segments):
        mask = [False] * num_items
        # choose (num_segments-1) cut points
        cuts = sorted(random.sample(range(1, num_items), num_segments - 1))
        seg_lengths, start = [], 0
        for c in cuts + [num_items]:
            seg_lengths.append(c - start)
            start = c
        return seg_lengths

    # Sample noise span lengths to sum to num_noise_tokens
    noise_span_lengths = _random_segmentation(num_noise_tokens, num_spans)
    # Sample non-noise span lengths to fill the rest
    num_nonnoise = length - num_noise_tokens
    nonnoise_span_lengths = _random_segmentation(num_nonnoise, num_spans + 1)

    # Interleave non-noise and noise spans, starting with non-noise
    spans = []
    for a, b in zip(nonnoise_span_lengths, noise_span_lengths + [0]):
        spans += [("nonnoise", a)]
        if b > 0:
            spans += [("noise", b)]
    # Build mask
    mask = []
    for kind, span_len in spans:
        mask += ([kind == "noise"] * span_len)
    # Truncate in case of rounding
    return mask[:length]

def _create_sentinel_ids(mask: List[bool], vocab: Dict[str, int]) -> List[int]:
    """Replace each masked span with a unique descending sentinel token id."""
    result = []
    sentinel_count = 0
    i = 0
    while i < len(mask):
        if mask[i]:
            sentinel_tok = f"<extra_id_{sentinel_count}>"
            result.append(vocab[sentinel_tok])
            sentinel_count += 1
            # Skip to end of this masked span
            i += 1
            while i < len(mask) and mask[i]:
                i += 1
        else:
            result.append(None)  # placeholder; fill with original token later
            i += 1
    return result

def _filter_and_fill(input_ids: List[int], mask: List[bool], sentinels_template: List[Optional[int]]):
    """Produce:
       - inputs: non-masked tokens with sentinel tokens at masked positions
       - targets: the concatenated masked spans each preceded by the matching sentinel
    """
    vocab = tokenizer.get_vocab()
    # Build inputs
    inputs = []
    i = 0
    sentinel_idx = 0
    while i < len(input_ids):
        if mask[i]:
            inputs.append(vocab[f"<extra_id_{sentinel_idx}>"])
            # Skip the entire masked span
            i += 1
            while i < len(input_ids) and mask[i]:
                i += 1
            sentinel_idx += 1
        else:
            inputs.append(input_ids[i])
            i += 1

    # Build targets: <extra_id_0> <masked-span-0> <extra_id_1> <masked-span-1> ...
    targets = []
    i = 0
    sentinel_idx = 0
    while i < len(input_ids):
        if mask[i]:
            targets.append(vocab[f"<extra_id_{sentinel_idx}>"])
            while i < len(input_ids) and mask[i]:
                targets.append(input_ids[i])
                i += 1
            sentinel_idx += 1
        else:
            i += 1
    targets.append(tokenizer.eos_token_id)  # end sequence
    return inputs, targets

@dataclass
class T5SpanCorruptionExample:
    input_ids: List[int]
    labels: List[int]
    attention_mask: List[int]

class CodeSpanDataset(Dataset):
    def __init__(self,
                 texts: List[str],
                 tokenizer: PreTrainedTokenizerFast,
                 max_length: int = 1024,
                 noise_density: float = 0.15,
                 mean_span_length: float = 3.0):
        self.texts = [t for t in texts if isinstance(t, str) and t.strip()]
        self.tok = tokenizer
        self.max_len = max_length
        self.noise_density = noise_density
        self.mean_span_length = mean_span_length
        self.vocab = tokenizer.get_vocab()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx) -> Dict[str, torch.Tensor]:
        txt = self.texts[idx]
        enc = self.tok(
            txt,
            truncation=True,
            max_length=self.max_len,
            return_attention_mask=True,
        )
        ids = enc["input_ids"]
        attn = enc["attention_mask"]

        # Make sure at least a few tokens are eligible
        if len(ids) < 8:
            # Return a dummy pass-through to avoid crashes; you may skip such samples earlier instead
            return {
                "input_ids": torch.tensor(ids),
                "labels": torch.tensor([tokenizer.eos_token_id]),
                "attention_mask": torch.tensor(attn),
            }

        # Build mask and corrupt
        mask = _random_spans_noise_mask(len(ids), self.noise_density, self.mean_span_length)
        inputs, targets = _filter_and_fill(ids, mask, None)

        # Final truncate (keep paired inputs/labels reasonable)
        inputs = inputs[:self.max_len]
        targets = targets[: self.max_len // 3]  # targets are shorter; adjust as needed

        return {
            "input_ids": torch.tensor(inputs, dtype=torch.long),
            "labels": torch.tensor(targets, dtype=torch.long),
            "attention_mask": torch.tensor([1] * len(inputs), dtype=torch.long),
        }


In [8]:
# Build datasets/splits
N = len(list_python_function)
train_texts = list_python_function[: int(N * 0.9)]  # keep a tiny slice for eval during pretrain
val_texts   = list_python_function[int(N * 0.9):]

train_ds = CodeSpanDataset(train_texts, tokenizer, max_length=1024, noise_density=0.15, mean_span_length=3.0)
val_ds   = CodeSpanDataset(val_texts, tokenizer, max_length=1024, noise_density=0.15, mean_span_length=3.0)

In [9]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding="longest",     
    label_pad_token_id=-100
)

In [10]:
# 5) TrainingArguments & Trainer
args = TrainingArguments(
    output_dir="t5-pretrain-ifspan",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=8,
    learning_rate=5e-4,
    weight_decay=0.01,
    num_train_epochs=3,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=1_000,
    eval_strategy="steps",
    eval_steps=1_000,
    save_steps=1_000,
    save_total_limit=3
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,  
    data_collator=data_collator
)

/tmp/ipykernel_3052549/2134485488.py:18: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [11]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 3}.
/home/tnguyen10/.conda/envs/ds_env/lib/python3.10/site-packages/transformers/data/data_collator.py:741: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)


Step,Training Loss,Validation Loss
1000,5.708900,4.805659


TrainOutput(global_step=1689, training_loss=5.308891228630107, metrics={'train_runtime': 1176.6109, 'train_samples_per_second': 22.947, 'train_steps_per_second': 1.435, 'total_flos': 6007841230110720.0, 'train_loss': 5.308891228630107, 'epoch': 3.0})

In [12]:
# Save final
trainer.save_model("t5-pretrain-ifspan/final")
tokenizer.save_pretrained("t5-pretrain-ifspan/final_tokenizer")

('t5-pretrain-ifspan/final_tokenizer/tokenizer_config.json',
 't5-pretrain-ifspan/final_tokenizer/special_tokens_map.json',
 't5-pretrain-ifspan/final_tokenizer/tokenizer.json')